Any cell that should be run everytime will have * in the top comment/line

near future upgrades :  
**implement limit of number of succesful licks (need to track licks first thooo**  
**alert when done, make it easy to see how much time has passed/is left**  
 *move arduino to opposite side of the rig to enable camera placement on user side *  
also i think we could make the lines specifically print around solenoid activated bc thats what i care about seeing real time (ie is it licking?)
   
resolved upgrade: buffered write to txt file (dont want to add delay if not needed    
how do you determine what the buffer size should be and what are the units ?      
caffeinate/prevent sleep based on OS. 
  
notes:   
it actuaLLY matters if mouse is holding bar when behavior starts, so we need to start behvaior before the bar is placed  
otherwise the bar touch sensor will not work until it slowly recalibrates itself   
  

tomorrow morning work on the damn camera system with atlas

Press esc M to change cell to text (markdown)
To stop cell, click outside of code section and press II quickly (not capital)   
To keyboard interrupt (ie stop cell stop behavior) click the leftside bar but outside the cell  
Press ii quickly

## new stuff to do 
- integrate analog signal with behavior code, make it a keyword in function/cell
- i think best thing is have fully seperate functions for behav alone, behav w 2p that are called by if systements after the series of input statements. 
- write function to send analog trigger and regular voltage pulse with recording into txt file
- update to sql instead of txt file????
- what if i preallocate a large pandas and trim after its done? is that an efficient mehtod?
- digital out from arduino, ideally goes to to both behav and atlas pc. can i use a split bnc? seems like the kind of thing our lab should definitely have. 
- where do those newly routed Di bnc cables go? ask gregg
- in the case where we dont get a split bnc, we can do arduino to nidaq/behav_pc and nidaq bnc back to atlas pc. would this require programming the nidaq outside of boston or would sending it via python be okay? or would it induce a lag. 

In [13]:
import serial
import time 
from datetime import datetime
import os
import csv
import pandas as pd
from tkinter import Tk
from tkinter.filedialog import askdirectory
import nidaqmx 
import time

(**mac**) To get port number type 'ls /dev/tty.*' in terminal

In [14]:
# get port info, on pc port should just be something like COM3
import serial.tools.list_ports
ports = serial.tools.list_ports.comports()
for port in ports:
    print(f"Port: {port.device}, Path: {port.hwid}, Description: {port.description}")

Port: COM3, Path: USB VID:PID=2341:0043 SER=851303035383512002A0 LOCATION=1-1, Description: Arduino Uno (COM3)
Port: COM1, Path: ACPI\PNP0501\0, Description: Communications Port (COM1)


In [15]:
#* caffeine!!! run to prevent sleep / interruption of data collection
import platform
import subprocess
import ctypes

os_name = os.name
if os_name =="nt":
    print("This is a windows system.")
   # Use ctypes to prevent sleep on Windows 
    ctypes.windll.kernel32.SetThreadExecutionState( 
        0x80000000 | 0x00000001 | 0x00000002 ) 
    print("Sleep mode disabled on Windows.")
elif os_name == "posix":
    if platform.system() == "Darwin":
        print("This is a macOS system.")
        caffeinate = subprocess.Popen(['caffeinate'])
        print("Caffeinate activated on macOS.")
    else:
        print("this is a Linux system.")

This is a windows system.
Sleep mode disabled on Windows.


### NIDAQ analog and digital trigger/counter functions

In [16]:
def send_analog_trigger(card_id='Dev1', channel='ao0', microscope = 'ATLAS', pulse_duration=0.001):
    """  Sends an analog trigger pulse (high then low) to the specified DAQ channel.
    
    Args:
    card_id (str): The ID of the DAQ device (e.g., 'Dev1').
    port_line (str): The port and line to which the signal is sent (e.g., 'port0/line1').
    pulse_duration (float): The duration (in seconds) for which the pulse stays high.
    """
    try:
        # Create a task to send the digital pulse
        with nidaqmx.Task() as task:
            # Add a digital output channel
            task.ao_channels.add_ao_voltage_chan(f'{card_id}/{channel}', name_to_assign_to_channel=f'AnI1_{microscope}')
            #set to 0, send 5V for pulse duration, return to 0
            task.write(0.0)
            task.write(5.0)
            time.sleep(pulse_duration) 
            task.write(0.0)
        print(f"Analog trigger sent to {card_id}/{channel} for {pulse_duration} seconds.")
        return True
    
    except Exception as e:
        print(f"Error sending analog trigger: {e}")
        return False
    return

### behavior functions

In [25]:
def run_behav(animal_ID, training_stage, min_, data_filepath, port='COM3', baudrate=115200):
    run_time = min_ * 60  # Convert minutes to seconds
    #initialize file to write to as 'file'
    with open(data_filepath, 'w') as file:
        #initialize serial connection
        ser = serial.Serial(port, baudrate, timeout=1)
        time.sleep(2) 
        #TRIGGER CAMERA, 2P HERE
        
        #send start command to Arduino
        start_time = time.time() #use for duration based arduino script start/stop
        ser.write(b'S') 
        behav_active = True
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
        print(f'{timestamp} - Sent start command to Arduino.')
        file.write(f'{timestamp} - Sent start command to Arduino.\n')
        
        try:
            count = 0
            # Read data from Arduino
            while behav_active:
                if ser.in_waiting > 0:
                    try:
                        line = ser.readline().decode('utf-8', errors='ignore').rstrip()
                        timestamp = datetime.now().strftime('%H:%M:%S.%f')[:-3]
                        log_line = f'{timestamp} - Arduino: {line}\n'
                        # Reduce print statements to every 10th message
                        count += 1
                        if count % 10 == 0:
                            print(log_line, end='')
                        #print(log_line, end='')  # Print the data in real-time
                        file.write(log_line)  # file.write() has an auto determined buffer size 
                        ##can implement specific buffering by calling the lines below every 100 measurements for example
                        #if count % 100 ==0:    
                            #file.flush()  # Ensure the data is written to the disk
                            #os.fsync(file.fileno())  # Force writing to disk
                    except UnicodeDecodeError:
                        print("Failed to decode serial data")
                        continue
        
                # Check if 20 seconds have passed to send the stop command
                if time.time() - start_time >= run_time:
                    #execution line (send stop command to arduino)
                    ser.write(b'X') 
                    #reporting lines
                    print('Sent stop command to Arduino.')
                    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
                    file.write(f'{timestamp} - Sent stop command to Arduino.')
                    #exit behavior code line
                    behav_active = False
        
        except KeyboardInterrupt:
            #execution line (send stop command to arduino)
            ser.write(b'X')
            #reporting lines
            print("Interrupted by user. Sent stop command to Arduino.")
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
            file.write(f'{timestamp} - Interrupted by user. Sent stop command to Arduino.')
            file.flush()
            os.fsync(file.fileno())
            behav_active = False
        finally:
            #close serial connection
            ser.close()
    return

consider adding additional error handling to write ser.close() to prevent com3 communication error (if serial line is opneed and not closed!)

In [18]:
def run_behav_w_imaging(animal_ID, training_stage, min_, data_filepath, microscope='ATLAS', port='COM3', baudrate=115200, card_id='Dev1', analog_channel='ao0', pulse_duration=0.500):
    run_time = min_ * 60  # Convert minutes to seconds
    #initialize file to write to as 'file'
    with open(data_filepath, 'w') as file:
        #initialize serial connection
        ser = serial.Serial(port, baudrate, timeout=1)
        time.sleep(2) 
        #TRIGGER CAMERA, 2P HERE
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
        send_analog_trigger(card_id='Dev1', channel='ao0', microscope = 'ATLAS', pulse_duration=0.500)
        print(f'{timestamp} - Sent analog trigger to {card_id}/{analog_channel} on {microscope}.')
        file.write(f'{timestamp} - Sent analog trigger to {card_id}/{analog_channel} on {microscope}.\n')
        
        #send start command to Arduino
        start_time = time.time() #use for duration based arduino script start/stop
        ser.write(b'S') 
        behav_active = True
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
        print(f'{timestamp} - Sent start command to Arduino.')
        file.write(f'{timestamp} - Sent start command to Arduino.\n')
        
        try:
            count = 0
            # Read data from Arduino
            while behav_active:
                if ser.in_waiting > 0:
                    try:
                        line = ser.readline().decode('utf-8', errors='ignore').rstrip()
                        timestamp = datetime.now().strftime('%H:%M:%S.%f')[:-3]
                        log_line = f'{timestamp} - Arduino: {line}\n'
                        # Reduce print statements to every 10th message
                        count += 1
                        if count % 10 == 0:
                            print(log_line, end='')
                        #print(log_line, end='')  # Print the data in real-time
                        file.write(log_line)  # file.write() has an auto determined buffer size 
                        ##can implement specific buffering by calling the lines below every 100 measurements for example
                        #if count % 100 ==0:    
                            #file.flush()  # Ensure the data is written to the disk
                            #os.fsync(file.fileno())  # Force writing to disk
                    except UnicodeDecodeError:
                        print("Failed to decode serial data")
                        continue
        
                # Check if 20 seconds have passed to send the stop command
                if time.time() - start_time >= run_time:
                    #execution line (send stop command to arduino)
                    ser.write(b'X') 
                    #reporting lines
                    print('Sent stop command to Arduino.')
                    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
                    file.write(f'{timestamp} - Sent stop command to Arduino.')
                    #exit behavior code line
                    behav_active = False
        
        except KeyboardInterrupt:
            #execution line (send stop command to arduino)
            ser.write(b'X')
            #reporting lines
            print("Interrupted by user. Sent stop command to Arduino.")
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
            file.write(f'{timestamp} - Interrupted by user. Sent stop command to Arduino.')
            file.flush()
            os.fsync(file.fileno())
            behav_active = False
            ser.close()
        finally:
            #close serial connection
            ser.close()
    return

### RUNTIME

in future, after imaging, if Y ask for microscope name, and fill in presaved params for NIDAQ channel. 

In [34]:
#request user for identiying information
animal_ID = input("Enter Animal ID: ")
training_stage = input("Enter training stage: ")
min_ = float(input("Session length (min): "))
imaging = input("2P Imaging? (Y/N): ")

#open window to choose directory 
root = Tk()
root.withdraw()  # Hide the main tkinter window
save_directory = askdirectory(title="Select Directory to Save Data File")
data_filename = f"{animal_ID}_{training_stage}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
data_filepath = os.path.join(save_directory, data_filename)

if imaging == "Y":
    run_behav_w_imaging(animal_ID, training_stage, min_, data_filepath, microscope='ATLAS',port='COM3', baudrate=115200)
if imaging == "N":
    run_behav(animal_ID, training_stage, min_, data_filepath, port='COM3', baudrate=115200)

Analog trigger sent to Dev1/ao0 for 0.5 seconds.
2024-12-13 19:17:47.220 - Sent analog trigger to Dev1/ao0 on ATLAS.
2024-12-13 19:17:47.724 - Sent start command to Arduino.
19:17:47.744 - Arduino: bar: 2
19:17:47.765 - Arduino: lick: 8
19:17:47.785 - Arduino: Solenoid pin state: 0
19:17:47.805 - Arduino: bar: 2
19:17:47.822 - Arduino: lick: 8
19:17:47.842 - Arduino: Solenoid pin state: 0
19:17:47.862 - Arduino: bar: 7
19:17:47.879 - Arduino: lick: 8
19:17:47.899 - Arduino: Solenoid pin state: 0
19:17:47.920 - Arduino: bar: 8
19:17:47.936 - Arduino: lick: 8
19:17:47.957 - Arduino: Solenoid pin state: 0
19:17:47.977 - Arduino: bar: 8
19:17:47.994 - Arduino: lick: 8
19:17:48.015 - Arduino: Solenoid pin state: 0
19:17:48.034 - Arduino: bar: 9
19:17:48.055 - Arduino: lick: 8
19:17:48.071 - Arduino: Solenoid pin state: 0
19:17:48.091 - Arduino: bar: 9
19:17:48.112 - Arduino: lick: 8
19:17:48.132 - Arduino: Solenoid pin state: 0
19:17:48.153 - Arduino: bar: 10
19:17:48.170 - Arduino: lick: 8

### end of session

In [ ]:
#**ALWAYS RUN to dump melatonin on the coffee jitter 
#jk but this cell is v important to reenable the computer running script to sleep
import platform
import subprocess
import ctypes
os_name = os.name
if os_name =="nt":
    print("This is a windows system.")
    # Re-enable sleep mode on Windows 
    ctypes.windll.kernel32.SetThreadExecutionState(0x80000000) 
    print("Sleep mode re-enabled on Windows. sweet dreams")
elif os_name == "posix":
    if platform.system() == "Darwin":
        print("This is a macOS system.")
        caffeinate.terminate()
        print("Caffeinate deactivated on macOS. sweet dreams")
    else:
        print("this is a Linux system.")


This is a windows system.
Sleep mode re-enabled on Windows. sweet dreams


### analog functions

In [ ]:
send_analog_trigger(card_id='Dev1', channel='ao0', microscope = 'ATLAS', pulse_duration=0.500)

Error sending analog trigger: name 'nidaqmx' is not defined


digital readout from behav pc and analog pulse to atlas should be the next upgrade, but will likely required threading (parallel processing) so lets come back to it later today!

In [ ]:
def send_analog_pulse(card_id='Dev1', channel='ao0', microscope = '2P3', trigger_duration = 0.500, pulse_duration=0.001, num_pulses = 50):
    """chatgpt written.
    Sends an analog trigger pulse (high then low) to the specified DAQ channel.
    
    Args:
    card_id (str): The ID of the DAQ device (e.g., 'Dev1').
    port_line (str): The port and line to which the signal is sent (e.g., 'port0/line1').
    pulse_duration (float): The duration (in seconds) for which the pulse stays high.
    """
    i=0
    try:
        # Create a task to send the digital pulse
        with nidaqmx.Task() as task:
            # Add a digital output channel
            task.ao_channels.add_ao_voltage_chan(f'{card_id}/{channel}', name_to_assign_to_channel=f'AnI1_{microscope}')
            
            # Send a analog high signal (1)
            task.write(0.0)
            task.write(5.0)
            time.sleep(trigger_duration)  # Wait for the specified duration

            # Send a digital low signal (0)
            task.write(0.0)
            print(f"Analog trigger sent to {card_id}/{channel} for {trigger_duration} seconds.")
        #send analog pulse sequence
            while i < num_pulses:
                task.write(5.0)
                time.sleep(pulse_duration)
                task.write(0.0)
                time.sleep(pulse_duration)
                i+=1
    except Exception as e:
        print(f"Error sending analog trigger: {e}")

In [ ]:
send_analog_pulse(card_id='Dev1', channel='ao0', microscope = 'ATLAS', trigger_duration = 0.500, pulse_duration=0.100, num_pulses = 50)

Error sending analog trigger: name 'nidaqmx' is not defined


### just test nidaq python connection. seems to be fine.

In [ ]:
def send_digital_output(card_id='Dev1', port_line='port0/line0', signal_pattern=[True, False], signal_duration=1):
    """
    Sends a digital output signal to a specified pin on the NI DAQ.

    Args:
        card_id (str): The ID of the NI DAQ device (e.g., 'Dev1').
        port_line (str): The digital output line to control (e.g., 'port0/line0').
        signal_pattern (list): List of signal states (e.g., [1, 0]) to send.
        signal_duration (float): Duration (in seconds) for each state in the pattern.
    """
    try:
        # Create a task for digital output
        with nidaqmx.Task() as task:
            # Add the specified digital output channel
            task.do_channels.add_do_chan(f"{card_id}/{port_line}")

            print(f"Sending digital output to {card_id}/{port_line}...")

            # Send the signal pattern
            while True:
                for state in signal_pattern:
                    task.write(state)  # Write the digital state (1 for HIGH, 0 for LOW)
                    print(f"Set {port_line} to {'HIGH' if state else 'LOW'}")
                    time.sleep(signal_duration)  # Hold the state for the specified duration

            print("Digital output complete.")

    except Exception as e:
        print(f"Error sending digital output: {e}")

In [ ]:
send_digital_output(card_id='Dev1', port_line='port1/line0', signal_pattern=[True, False], signal_duration=0.1)

In [ ]:
#test in jupyter lab rather than jupyter notebook. delete if doesnt work. 
import ipywidgets as widgets
from IPython.display import display

# Create a dropdown for selecting whether imaging is enabled or not
imaging_dropdown = widgets.Dropdown(
    options=[(False, 'No Imaging'), (True, 'With Imaging')],
    value=False,  # Default selection
    description='Imaging:',
)

# Display the dropdown
display(imaging_dropdown)

# Retrieve the selected value (True or False)
imaging = imaging_dropdown.value

# If imaging is enabled, send a digital trigger
if imaging:
    send_digital_trigger()  # Calls the function to send the digital signal

# Proceed with starting behavior by sending 'S' to Arduino
ser.write(b'S')
